# EvoFlow use case: NK-cell signalling knock-outs

This notebook is an end-to-end vignette for **EvoFlow**. It was written to run
inside the Jupyter container published with the project:

```bash
docker run --rm -p 8888:8888 ghcr.io/vergaju/evoflow:latest
```

That image already contains Python, **NicheNetPy**, **pertpy** and EvoFlow, so
no local installation is required. Open `vignettes/use_case.ipynb` inside
JupyterLab and run all cells.

The workflow mirrors the reference pipeline:

1. **NicheNetPy** — infer a weighted ligand-to-target signalling network from
   a short table of receptors and target transcription factors.
2. **Expression data** — load the `kang_2018` PBMC dataset with `pertpy`, keep
   the NK cells and build the samples x genes matrix used to weight capacities.
3. **EvoFlow** — prepare the flow network and run the genetic algorithm.
4. **Interpretation** — compare the balanced, flow, cost and redundancy
   solutions.

> **First run.** NicheNetPy downloads its reference model from Zenodo (about
> 600 MB) and `pertpy` downloads the `kang_2018` dataset (about 38 MB). Both
> are cached for later runs.
>
> **Figures.** Plotting is fixed at 96 DPI with small figure sizes so the
> committed notebook stays lightweight in git.

## 0. Setup

In [ ]:
import os
import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

import EvoFlow

# Keep every figure small and low-resolution (the notebook is committed to git).
plt.rcParams["figure.dpi"] = 96
plt.rcParams["savefig.dpi"] = 96
plt.rcParams["figure.figsize"] = (6, 4)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

RANDOM_SEED = 0
np.random.seed(RANDOM_SEED)

print("EvoFlow:", EvoFlow.__version__)
print("Python:", sys.version.split()[0])
print("Working directory:", Path.cwd())

## 1. Infer the signalling network with NicheNetPy

NicheNet needs two short lists:

- **receptors** expressed by the receiver cells (here NK cells),
- **targets**, i.e. transcription factors whose downstream activity we want to
  modulate.

Candidate ligands are the ligands in NicheNet's ligand-receptor network whose
receptor is in our list. NicheNetPy then keeps the highest-weight signalling
paths from those ligands to the requested targets and returns the ligand ->
signalling -> gene-regulatory subnetwork.

In [ ]:
WORKDIR = Path.cwd()
INPUT_DIR = WORKDIR / "inputs"
OUTPUT_DIR = WORKDIR / "outputs"
NICHENET_DIR = OUTPUT_DIR / "nichenet"
for directory in (INPUT_DIR, OUTPUT_DIR, NICHENET_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Receptors present in NicheNet's ligand-receptor network and relevant to NK
# cells, plus NK-lineage transcription factors used as targets.
RECEPTORS = ["IL2RB", "IL2RG", "IL15RA", "KLRK1", "IFNGR1", "IL12RB1"]
TARGETS = ["TBX21", "EOMES", "STAT1", "IRF1", "STAT4", "PRDM1"]

SAMPLE = "nk_signalling"
inputs = pd.DataFrame({"receptors": RECEPTORS, "targets": TARGETS})
inputs.to_csv(INPUT_DIR / f"{SAMPLE}.csv", index=False)
inputs

Download and cache the NicheNet reference model. The ligand-target matrix and
the weighted ligand-signalling / gene-regulatory networks are published on
Zenodo and are reused on later runs.

In [ ]:
from urllib.request import urlopen

REFERENCE_FILES = {
    "ltf_matrix_human.pkl": (
        "https://zenodo.org/records/17061000/files/ltf_matrix_human.pkl"
    ),
    "nichenet_human.pkl": (
        "https://zenodo.org/records/17061000/files/nichenet_human.pkl"
    ),
}
reference_dir = NICHENET_DIR / "reference"
reference_dir.mkdir(parents=True, exist_ok=True)

for filename, url in REFERENCE_FILES.items():
    destination = reference_dir / filename
    if destination.exists():
        continue
    print(f"downloading {filename} ...")
    partial = destination.with_suffix(destination.suffix + ".part")
    with urlopen(url, timeout=600) as response, open(partial, "wb") as handle:
        while chunk := response.read(1 << 20):
            handle.write(chunk)
    partial.replace(destination)
    print(f"saved {destination} ({destination.stat().st_size / 1e6:.0f} MB)")

Load the reference model, derive the candidate ligands from the selected
receptors and run `get_ligand_signaling_path`. The last step builds the final
weighted network and the node annotation table consumed by EvoFlow.

In [ ]:
from nichenetpy.visualization import get_ligand_signaling_path

with open(reference_dir / "ltf_matrix_human.pkl", "rb") as handle:
    ltf = pickle.load(handle)
with open(reference_dir / "nichenet_human.pkl", "rb") as handle:
    model = pickle.load(handle)

lr_sig = pd.DataFrame(model["lr_sig"]._mapping, columns=["from", "to", "weight"])
gr = pd.DataFrame(model["gr"]._mapping, columns=["from", "to", "weight"])
lr_network = pd.DataFrame(model["lr_network"]._mapping, columns=["from", "to"])

ligands = sorted(set(lr_network.loc[lr_network["to"].isin(RECEPTORS), "from"]))
gr_targets = set(gr["to"].unique())
targets = [target for target in TARGETS if target in gr_targets]
print(f"{len(ligands)} candidate ligands: {ligands}")
print(f"targets kept: {targets}")
if not ligands or not targets:
    raise RuntimeError(
        "No ligands or targets remain for the selected receptors. "
        "Adjust RECEPTORS/TARGETS."
    )

In [ ]:
tf_signaling, tf_regulatory = get_ligand_signaling_path(
    ltf["mat"],
    ligands,
    targets,
    all_ligands=ltf["row_names"],
    all_targets=ltf["col_names"],
    lr_sig=lr_sig,
    gr=gr,
    top_n_regulators=4,
    minmax_scaling=False,
)

network_df = pd.concat(
    [
        tf_signaling.assign(layer="signaling"),
        tf_regulatory.assign(layer="regulatory"),
    ],
    ignore_index=True,
)
network_nodes = set(network_df["from"]) | set(network_df["to"])
print(f"net: {len(network_df)} edges, {len(network_nodes)} nodes")
print("receptors in network:", [r for r in RECEPTORS if r in network_nodes])
display(network_df.head())

Annotate every node for EvoFlow's plotting and diagnostics. Receptors come from
the ligand-receptor network, transcriptional regulators from the gene
regulatory network, and the remaining nodes are signalling mediators.

In [ ]:
receptor_genes = set(lr_network["to"])
regulator_genes = set(gr["from"])
ligand_set, target_set = set(ligands), set(targets)

rows = []
for gene in sorted(network_nodes):
    if gene in ligand_set:
        annotation = "ligand"
    elif gene in target_set:
        annotation = "target"
    elif gene in receptor_genes:
        annotation = "receptor"
    elif gene in regulator_genes:
        annotation = "transcriptional regulator"
    else:
        annotation = "signaling mediator"
    rows.append({"gene": gene, "annotation": annotation})
annotation_df = pd.DataFrame(rows)

network_path = NICHENET_DIR / f"{SAMPLE}_weighted_signaling_network.tsv"
annotation_path = NICHENET_DIR / f"{SAMPLE}_annotation_table.tsv"
network_df.to_csv(network_path, sep="\t", index=False)
annotation_df.to_csv(annotation_path, sep="\t", index=False)

print("saved", network_path.name, "and", annotation_path.name)
display(annotation_df["annotation"].value_counts())

## 2. Expression data: NK cells from `kang_2018`

EvoFlow converts expression into one **percolation** value per gene, which
scales edge capacities. We use `pertpy` to download the Kang et al. (2018)
PBMC dataset, keep the `NK cells` and restrict the matrix to the genes present
in the NicheNet annotation. Subsetting before densifying keeps the matrix small
(the full dataset is about 25k cells x 16k genes).

In [ ]:
import pertpy as pt

adata = pt.dt.kang_2018()
print(adata)
display(adata.obs["cell_type"].value_counts())

In [ ]:
annotation_genes = set(annotation_df["gene"].astype(str)) & set(adata.var_names)
genes = sorted(annotation_genes)
print(
    f"{len(genes)} of {annotation_df['gene'].nunique()} annotated genes are "
    "measured in kang_2018"
)

nk = adata[adata.obs["cell_type"].astype(str) == "NK cells", genes].copy()
nk.obs["condition"] = nk.obs["label"].astype(str)
print("NK cells:", nk.n_obs, "| genes:", nk.n_vars)
display(nk.obs["condition"].value_counts())

values = nk.X.toarray() if hasattr(nk.X, "toarray") else np.asarray(nk.X)
expression_df = pd.DataFrame(values, index=nk.obs_names, columns=nk.var_names)
expression_path = INPUT_DIR / "kang_2018_nk_expression.csv"
expression_df.to_csv(expression_path)
print("saved", expression_path, expression_df.shape)

In [ ]:
markers = [gene for gene in ["NKG7", "GNLY", "PRF1", "GZMB"] if gene in nk.var_names]
if markers:
    sc.pl.violin(nk, markers, groupby="condition", show=False, stripplot=False)
    plt.gcf().set_size_inches(6, 3)
    plt.show()

## 3. Run EvoFlow on the inferred network

The low-level API mirrors the reference pipeline: load the network,
annotation and expression matrix, compute percolation, build the graph, update
capacities and prepare the `SuperSource`/`SuperSink` topology. `lock_targets=1`
protects the original receptor nodes from removal.

In [ ]:
from EvoFlow.biology import BioNetworkPreparer

preparer = BioNetworkPreparer(lock_targets=1)
preparer.load_data(
    network_path=str(network_path),
    annotation_path=str(annotation_path),
    expression_path=str(expression_path),
)
preparer.compute_percolation(stat="nonzero_ratio")
preparer.build_base_network()
preparer.update_edge_capacities(method="multiplicative")

# Keep only the receptors/targets that made it into the inferred network.
receptors = [gene for gene in RECEPTORS if gene in preparer.G]
sinks_genes = [gene for gene in TARGETS if gene in preparer.G]
print("sources:", receptors)
print("sinks:", sinks_genes)
if not receptors or not sinks_genes:
    raise RuntimeError(
        "The NicheNet network contains none of the selected receptors/targets. "
        "Inspect the annotation table and adjust RECEPTORS/TARGETS."
    )

sources, sinks = preparer.prepare_bioflow_topology(
    receptors=receptors, target_genes=sinks_genes
)
network = preparer.G
print(f"graph: {network.number_of_nodes()} nodes, {network.number_of_edges()} edges")

The genetic algorithm removes a small budget of nodes (5% of the network by
default here) while minimising flow, removal cost and SCC redundancy. Parameters
are set explicitly so the run is short and reproducible; omit
`population_size`/`early_stop_gen` to let the bundled GAM choose them.

EvoFlow writes its artifacts (`ga_run_*.jsonl`) to the current working
directory, so we run it inside `outputs/evoflow`.

In [ ]:
from EvoFlow.core import GAOptimizer

budget = max(2, int(round(0.05 * network.number_of_nodes())))
print("removal budget:", budget)

optimizer = GAOptimizer(
    network,
    sources,
    sinks,
    max_remove=budget,
    delta_penalty=False,
    repair=True,
    dynamic=True,
    use_multiprocessing=False,  # deterministic and safe inside Jupyter
)

evoflow_dir = OUTPUT_DIR / "evoflow"
evoflow_dir.mkdir(parents=True, exist_ok=True)
previous_cwd = Path.cwd()
os.chdir(evoflow_dir)
try:
    results, logs, population, norm = optimizer.run(
        generations=200,
        population_size=50,
        early_stop_gen=20,
        guess_parameters=False,
        verbose=True,
    )
finally:
    os.chdir(previous_cwd)
print("optimization complete")

## 4. Inspect the results

In [ ]:
rows = []
for solution, payload in results.items():
    row = {
        "solution": solution,
        "num_removed": payload["num_removed"],
        **{name: round(value, 3) for name, value in payload["fitness"].items()},
        "removed_nodes": ", ".join(payload["removed_nodes"]),
    }
    rows.append(row)
summary = pd.DataFrame(rows).set_index("solution")
display(summary)

In [ ]:
logs_df = pd.DataFrame(logs).T.reset_index(names="gen").sort_values("gen")
logs_df["gen"] = logs_df["gen"].astype(int)

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, metric in zip(axes, ["Flow", "Redundancy"]):
    if metric in logs_df:
        ax.plot(logs_df["gen"], logs_df[metric], linewidth=1.2)
        ax.set_title(metric)
        ax.set_xlabel("generation")
        ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
from EvoFlow.visualization import plot_graph

final_graph = optimizer.SG.copy()
final_graph.remove_nodes_from(results["Best_balanced"]["removed_nodes"])
fig, ax = plot_graph(final_graph, filename_prefix=None, layout="spring", figsize=(6, 4))
plt.show()

### How to read the table

- **Flow** is the remaining maximum flow after removing the selected nodes;
  lower is better.
- **Node** is the number of removed nodes (the cost of the intervention).
- **Redundancy** counts strongly connected components that remain after the
  removal, i.e. residual redundant signalling.
- `Best_balanced` is the Pareto solution closest to the origin after
  per-objective normalisation, `Best_Flow` minimises flow, `Best_Cost`
  minimises the number of removed nodes and `Best_Redundancy` minimises
  redundancy.

Because targets (and, with `lock_targets=1`, receptors) are protected, the
candidates sit on the signalling mediators between the NK receptors and the
transcription factors. In an experimental follow-up you would prioritise the
genes that appear in `Best_balanced` and are also differentially expressed
between conditions.

## Alternative: the one-call CLI / Python API

For production runs the same pipeline is available through
`EvoFlow.cli.run_evoflow` (and the `evoflow` console script). It also handles
h5ad cell-type filtering, bulk expression files and automatic output naming:

```python
from EvoFlow.cli import run_evoflow

run_evoflow(
    network=str(network_path),
    annotation=str(annotation_path),
    expression_file=str(expression_path),
    receptors_raw="IL2RB|IL2RG|IL15RA",
    mediators_raw="TBX21,EOMES,STAT1",
    ligands_file="ligands.csv",   # CSV with an 'x' column of ligand genes
    targets="TBX21,EOMES,STAT1",  # alternative to a regulons file
    generations=2000,
)
```

Use the low-level API shown above when you already have pre-processed inputs
or want to keep the `GAOptimizer` object (for example to inspect the active
subgraph or to pickle it for later runs).

## Saving the executed notebook

The committed version of this notebook intentionally contains **no outputs**.
To publish it with results:

1. build/pull the container described in the documentation,
2. run `vignettes/use_case.ipynb` from top to bottom in JupyterLab,
3. save it (`File > Save and Checkpoint`) and commit the `.ipynb` before
   tagging the release.

All figures are already configured at 96 DPI with small figure sizes, and the
only downloaded data are cached outside the repository.